# Saving Waveforms as NPZ and MAT Files

**Scott Prahl**

**Apr 2026**

In [ ]:
%config InlineBackend.figure_format = 'retina'

import os
import struct
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from RigolWFM import Wfm

repo = "https://raw.githubusercontent.com/scottprahl/RigolWFM/main/tests/files/"

def sample_url(relative_path: str) -> str:
    return repo + relative_path

def _pad8(length: int) -> int:
    return (-length) % 8

def load_simple_mat_v5(source: bytes | bytearray | str | Path) -> dict[str, np.ndarray]:
    if isinstance(source, (str, Path)):
        payload = Path(source).read_bytes()
    else:
        payload = bytes(source)

    if len(payload) < 128 or payload[126:128] != b'IM':
        raise ValueError('Only little-endian MATLAB v5 files are supported.')

    out: dict[str, np.ndarray] = {}
    offset = 128
    while offset + 8 <= len(payload):
        data_type, size = struct.unpack_from('<II', payload, offset)
        offset += 8
        if data_type == 0 and size == 0:
            break
        if data_type != 14:
            raise ValueError(f'Unexpected MAT element type {data_type}.')

        matrix = payload[offset : offset + size]
        offset += size + _pad8(size)
        inner = 0

        def read_element(expected_type: int) -> bytes:
            nonlocal inner
            elem_type, elem_size = struct.unpack_from('<II', matrix, inner)
            inner += 8
            if elem_type != expected_type:
                raise ValueError(f'Unexpected MAT subelement type {elem_type}.')
            data = matrix[inner : inner + elem_size]
            inner += elem_size + _pad8(elem_size)
            return data

        _flags = read_element(6)
        dims_raw = read_element(5)
        dims = struct.unpack('<' + ('i' * (len(dims_raw) // 4)), dims_raw)
        name = read_element(1).decode('ascii')
        real_type, real_size = struct.unpack_from('<II', matrix, inner)
        inner += 8
        real = matrix[inner : inner + real_size]

        if real_type == 9:
            array = np.frombuffer(real, dtype='<f8').copy()
        elif real_type == 2:
            array = np.frombuffer(real, dtype=np.uint8).copy()
        else:
            raise ValueError(f'Unsupported MAT real data type {real_type}.')

        if dims:
            shape = tuple(int(dim) for dim in dims)
            array = array.reshape(shape, order='F')
            if array.shape == (1, 1):
                array = array.reshape(1)
            elif len(array.shape) == 2 and array.shape[1] == 1:
                array = array[:, 0]
        out[name] = array

    return out

## Introduction

`RigolWFM` can export parsed waveforms as either a NumPy archive (`.npz`) or a MATLAB v5 file (`.mat`).  Both formats carry the same basic arrays:

* `time` — one timestamp per sample
* `start` — the first timestamp
* `increment` — the uniform sample spacing
* one array per exported channel, such as `CH_1`, `CH_2`, or `D6`

This notebook uses real sample files from the GitHub repo and shows both an analog and a digital export.

## NPZ Export From an Analog Capture

We will start with a two-channel DS1102E waveform and export it to a NumPy archive.

In [ ]:
analog = Wfm.from_url(sample_url('wfm/DS1102E-D.wfm'))
print(analog.describe())

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    npz_path = Path(tmpdir) / 'DS1102E-D.npz'
    analog.npz(npz_path)
    with np.load(npz_path) as archive:
        npz_keys = sorted(archive.files)
        npz_summary = {name: archive[name].shape for name in npz_keys}
        npz_start = float(archive['start'][0])
        npz_increment = float(archive['increment'][0])

npz_keys, npz_summary, npz_start, npz_increment

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    npz_path = Path(tmpdir) / 'DS1102E-D.npz'
    analog.npz(npz_path)
    with np.load(npz_path) as archive:
        t = archive['time']
        ch1 = archive['CH_1']
        ch2 = archive['CH_2']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t[:2000] * 1e3, ch1[:2000], label='CH 1')
ax.plot(t[:2000] * 1e3, ch2[:2000], label='CH 2')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Voltage (V)')
ax.set_title('First 2000 NPZ Samples From DS1102E-D.wfm')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## MAT Export From a Digital Capture

The same export path also works for parsed logic traces.  Here we export a DS1074Z waveform whose active digital line is `D6`.

In [ ]:
logic = Wfm.from_url(sample_url('wfm/DS1074Z-C.wfm'), model='Z')
print(logic.describe())

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    mat_path = Path(tmpdir) / 'DS1074Z-C.mat'
    logic.mat(mat_path)
    mat_data = load_simple_mat_v5(mat_path)

sorted(mat_data), {name: mat_data[name].dtype.name for name in sorted(mat_data)}, {name: mat_data[name].shape for name in sorted(mat_data)}

In [ ]:
time = mat_data['time']
d6 = mat_data['D6']

fig, ax = plt.subplots(figsize=(10, 3))
ax.step(time * 1e3, d6, where='post', color='tab:red')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('State')
ax.set_xlim(41,42)
ax.set_yticks([0, 1])
ax.set_ylim(-0.2, 1.2)
ax.set_title('Samples From DS1074Z-C.wfm')
ax.grid(True, alpha=0.3)
plt.show()

## Command-Line Equivalents

The same exports are also available from `wfmconvert`:

```bash
wfmconvert npz DS1102E-D.wfm
wfmconvert mat DS1074Z-C.wfm
```

The array names inside the files match the parsed channel names after light sanitization for identifier-like formats, so `CH 1` becomes `CH_1`, while digital names such as `D6` are already ready to use.